# S6E5 — Colab runner

Generic notebook for running any `notebooks/0X_*.py` training script on Colab Pro with GPU.

**Workflow** (run cells top to bottom):
1. Mount Google Drive (Drive folder `s6e5/` is the persistent artifact store)
2. Clone GitHub repo (latest code from `main`)
3. **Version meta** — verify which script will run (edit `SCRIPT` here)
4. Install dependencies (just-in-time, only what current version needs)
5. Sync data Drive → /content
6. Run the target script
7. Sync artifacts back to Drive
8. (optional) Download submission CSV to local machine

**Drive folder**: https://drive.google.com/drive/folders/1N6PFShEtMj2KSYWxaQQz-6Kro1CTLilh

**Verified path** (Drive MCP, 2026-05-12): `/content/drive/MyDrive/Colab Notebooks/kaggle/s6e5`

**Expected structure**:
```
MyDrive/Colab Notebooks/kaggle/s6e5/
├── data/
│   ├── raw/{train.csv, test.csv, sample_submission.csv}   ← MUST be real .csv, NOT Sheets
│   └── external/f1_strategy_dataset_v4.csv
├── probs/                                                  (auto-created)
├── submissions/                                            (auto-created)
└── experiments.jsonl                                       (synced from Colab runs)
```

⚠ **If Drive auto-converted the CSVs to Google Sheets** (files without `.csv` extension / green Sheets icon), the script will fail to read them. Fix: <https://drive.google.com/drive/settings> → uncheck **"Convert uploaded files to Google Docs editor format"** → delete + re-upload.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verified via Drive MCP 2026-05-12: folder lives at MyDrive/Colab Notebooks/kaggle/s6e5
# Note the space in 'Colab Notebooks' — DO NOT remove it; this is Google's auto-created folder name.
DRIVE_S6E5 = '/content/drive/MyDrive/Colab Notebooks/kaggle/s6e5'

!ls -la "$DRIVE_S6E5" || echo 'PATH NOT FOUND — adjust DRIVE_S6E5 above.'

# Sanity: data/raw/*.csv must be REAL csv files, not Google Sheets.
# If you see no '.csv' extension, the upload was converted to Sheets:
# fix at https://drive.google.com/drive/settings → uncheck 'Convert uploads' → re-upload.

# --- Kaggle CLI auth from Colab userdata secrets ---
# Reads KAGGLE_USERNAME + KAGGLE_API_TOKEN from Colab "Secrets" panel
# (set them once via the key icon on the left sidebar).
#
# Defensive: if KAGGLE_API_TOKEN happens to be the full kaggle.json content
# (the {"username":"...","key":"..."} blob), auto-extract the "key" field.
import os, json
try:
    from google.colab import userdata
    username = userdata.get('KAGGLE_USERNAME')
    api_token = userdata.get('KAGGLE_API_TOKEN')

    # Auto-unwrap JSON if pasted as kaggle.json content
    if api_token and api_token.strip().startswith('{'):
        try:
            parsed = json.loads(api_token)
            if isinstance(parsed, dict):
                if 'key' in parsed:
                    api_token = parsed['key']
                if 'username' in parsed and not username:
                    username = parsed['username']
                print("ℹ KAGGLE_API_TOKEN looked like JSON — auto-extracted 'key' field")
        except json.JSONDecodeError:
            pass

    if not username or not api_token:
        raise ValueError("missing username or token")

    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = api_token
    print(f"✓ Kaggle auth set (user: {username}, key length: {len(api_token)})")
except Exception as exc:
    print(f"⚠ Kaggle auth skipped: {exc}")
    print("  Set KAGGLE_USERNAME + KAGGLE_API_TOKEN in Colab Secrets, or upload kaggle.json manually.")

## 2. Clone repo + check out latest

In [ ]:
%cd /content
!rm -rf playground-s6e5
!git clone -q https://github.com/SirGrigor/playground-s6e5.git
%cd playground-s6e5
!git log -1 --oneline

## 3. Version meta — verify which script will run

Edit `SCRIPT` below to pick which model version to train. Re-run this cell to see the verification block before consuming Colab compute.

In [ ]:
# === VERSION META — verify what we're about to run ===
# Default: one-shot pipeline with pre-flight validation.
# Validates ALL data prerequisites BEFORE any work. Fails fast with clear messages.

SCRIPT = 'notebooks/25_pipeline.py'
EXTRA_ARGS = ''   # optional: '--baseline v18.002' to override auto-detection

print("=" * 70)
print(f"ABOUT TO RUN:  {SCRIPT}")
print(f"EXTRA ARGS:    {EXTRA_ARGS or '(none)'}")
print("=" * 70)
print()
print("--- Latest commit on this branch ---")
!git log -1 --oneline
!git log -1 --format='committed %cd' --date=relative
print()
print(f"--- {SCRIPT} header (first 30 lines) ---")
!head -30 {SCRIPT}
print()
print("=" * 70)
print("If this isn't the version you intended → edit SCRIPT above, re-run THIS cell.")
print("Otherwise → continue with the cells below.")
print("=" * 70)

## 4. Install dependencies

Just-in-time per version. v1 (LGB) and v2 (LGB+TE) only need `pyarrow` — Colab pre-installs the rest. v3/v5 will install catboost / pytabkit when they're needed.

In [ ]:
# Default to uv (per feedback_uv-over-pip). ONE pip call bootstraps uv,
# everything else uses uv pip.
#
# Detect dependencies by scanning the target script's imports — robust to
# future script names. Old keyword-matching version missed v14b/v15.

import os
from pathlib import Path

SCRIPT_LOCAL = SCRIPT if 'SCRIPT' in dir() else ''
script_text = Path(SCRIPT_LOCAL).read_text() if SCRIPT_LOCAL and Path(SCRIPT_LOCAL).exists() else ''
script_lower = script_text.lower()

NEEDS_PYTABKIT = 'pytabkit' in script_lower or 'realmlp_td' in script_lower or 'tabm_d' in script_lower
NEEDS_MVF = 'ml_variant_factory' in script_lower or 'ml-variant-factory' in script_lower
NEEDS_TABPFN = 'tabpfn' in script_lower or 'finetunedtabpfn' in script_lower

MVF_VERSION = "v0.1.2"   # bump this to pull a newer release

!pip install -q uv

extras = ["pyarrow"]
if NEEDS_PYTABKIT:
    extras.append("pytabkit")
if NEEDS_TABPFN:
    extras.append("tabpfn")
    print("Installing pytabkit (RealMLP/TabM) — pulls torch+lightning+numpy upgrades.")
    print("If you ran v1-v6 in this same session, RESTART RUNTIME before this cell")
    print("to avoid C ABI mismatch on sklearn.\n")

!uv pip install -q --system {' '.join(extras)}

if NEEDS_MVF:
    print(f"Installing ml-variant-factory @ {MVF_VERSION} from GitHub...")
    !uv pip install -q --system "git+https://github.com/SirGrigor/ml-variant-factory@{MVF_VERSION}"

# Verify imports
import lightgbm, sklearn, numpy, pandas, pyarrow
print(f'numpy {numpy.__version__}  pandas {pandas.__version__}  sklearn {sklearn.__version__}')
print(f'lgb {lightgbm.__version__}  pyarrow {pyarrow.__version__}')

if NEEDS_PYTABKIT:
    import pytabkit
    from importlib.metadata import version
    print(f'pytabkit {version("pytabkit")}')

if NEEDS_MVF:
    from importlib.metadata import version
    import ml_variant_factory as _mvf
    print(f'ml-variant-factory {version("ml-variant-factory")}')

import xgboost
print(f'xgb {xgboost.__version__}')

assert tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1, 3), \
    f"sklearn {sklearn.__version__} is too old — need >= 1.3 for CV-safe TargetEncoder"
print("\n✓ sklearn version supports CV-safe TargetEncoder")

import torch
print(f'\ntorch {torch.__version__}  CUDA={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

if NEEDS_TABPFN:
    import tabpfn
    from importlib.metadata import version
    print(f"tabpfn {version('tabpfn')}")
    import os
    if not os.environ.get("TABPFN_TOKEN"):
        print("\n⚠ TABPFN_TOKEN not set. The fit step will fail.")
        print("  Either set os.environ[\"TABPFN_TOKEN\"] in a prior cell")
        print("  OR retrieve from Kaggle secrets if running on Kaggle notebook.")


## 5. Sync data Drive → /content

Drive IO is slow — copy once per Colab session.

In [ ]:
import os, shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/external', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
os.makedirs('probs', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

# raw competition data
for fn in ('train.csv', 'test.csv', 'sample_submission.csv'):
    src = f'{DRIVE_S6E5}/data/raw/{fn}'
    dst = f'data/raw/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')
    else:
        print(f'  already present: {fn}')

# external dataset
for fn in ('f1_strategy_dataset_v4.csv',):
    src = f'{DRIVE_S6E5}/data/external/{fn}'
    dst = f'data/external/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')
    else:
        print(f'  already present: {fn}')

# sacred holdout split is in git already (data/splits/holdout_v1.parquet) — no copy needed
print('\n--- Verifying files ---')
!ls -la data/raw data/external data/splits

## 6. Run the target script

Uses `SCRIPT` and `EXTRA_ARGS` from the **Version meta** cell above. Don't edit anything here — edit Version meta then come back.

In [ ]:
print(f"RUNNING: {SCRIPT}  {EXTRA_ARGS}")
print(f"cwd: {os.getcwd()}")
!python {SCRIPT} {EXTRA_ARGS}

## 7. Sync artifacts /content → Drive

Persist outputs back so they survive Colab session shutdown.

In [ ]:
import os, shutil

synced_any = False

# probs/ — model artifacts (oof/holdout/test .npy per version)
if os.path.isdir('probs'):
    for d in sorted(os.listdir('probs')):
        src = f'probs/{d}'
        dst = f'{DRIVE_S6E5}/probs/{d}'
        os.makedirs(dst, exist_ok=True)
        for fn in os.listdir(src):
            shutil.copyfile(f'{src}/{fn}', f'{dst}/{fn}')
        print(f'  synced probs/{d}/ → Drive')
    synced_any = True

# submissions/ — final CSVs for Kaggle
if os.path.isdir('submissions'):
    os.makedirs(f'{DRIVE_S6E5}/submissions', exist_ok=True)
    for fn in sorted(os.listdir('submissions')):
        shutil.copyfile(f'submissions/{fn}', f'{DRIVE_S6E5}/submissions/{fn}')
        print(f'  synced submissions/{fn} → Drive')
    synced_any = True

# harvest/ — public-OOF + submission downloads (can be ~1-2 GB for v18 harvest)
# Use rsync-style incremental copy for speed (skip identical files via dirs_exist_ok + copytree).
if os.path.isdir('harvest'):
    for tier in sorted(os.listdir('harvest')):  # e.g., 'v13', 'v18'
        src = f'harvest/{tier}'
        dst = f'{DRIVE_S6E5}/harvest/{tier}'
        if not os.path.isdir(src):
            continue
        os.makedirs(dst, exist_ok=True)
        # Top-level files (manifest.json, audit.md, etc.)
        for fn in os.listdir(src):
            src_path = f'{src}/{fn}'
            dst_path = f'{dst}/{fn}'
            if os.path.isfile(src_path):
                shutil.copyfile(src_path, dst_path)
            elif os.path.isdir(src_path):
                if not os.path.exists(dst_path):
                    shutil.copytree(src_path, dst_path)
                # else: kernel subdir already on Drive — skip (avoid re-uploading large files)
        print(f'  synced harvest/{tier}/ → Drive')
    synced_any = True

# experiments.jsonl — append-only experiment log
if os.path.exists('experiments.jsonl'):
    shutil.copyfile('experiments.jsonl', f'{DRIVE_S6E5}/experiments.jsonl')
    print('  synced experiments.jsonl → Drive')
    synced_any = True

# releases.jsonl — progressive-blend release log (per session)
if os.path.exists('releases.jsonl'):
    shutil.copyfile('releases.jsonl', f'{DRIVE_S6E5}/releases.jsonl')
    print('  synced releases.jsonl → Drive')
    synced_any = True

if not synced_any:
    print('ABORT: nothing to sync. The run script in cell 6 did not produce any artifacts.')
    print('Check the run cell output for errors before re-running this cell.')
else:
    print('\n✓ sync complete.')

## 8. Download submission CSV to local machine

Use this for the FINAL submission of each version. The submission then goes via `kaggle competitions submit` from your local machine.

In [ ]:
# Auto-pick the winning submission and download to local.
# Pipeline philosophy: cell 6 produces multiple v18.NNN releases,
# cell 7 syncs ALL of them to Drive, cell 8 downloads only the winner.

import os
from pathlib import Path
from google.colab import files

all_subs = sorted(Path('submissions').glob('*.csv'))
print(f"Available submissions ({len(all_subs)}):")
for p in all_subs:
    size_kb = p.stat().st_size // 1024
    print(f"  {p.name}  ({size_kb} KB)")
print()

# Pick the latest v18.NNN release (highest number = most recently promoted by progressive blend)
v18_releases = sorted([p for p in all_subs if p.name.startswith('v18.')])
if v18_releases:
    default = v18_releases[-1].name
    print(f"WINNER (latest v18 release): {default}")
elif all_subs:
    try:
        stem = Path(SCRIPT).stem.split('_', 1)[1]
        default = f"{stem}.csv"
    except Exception:
        default = all_subs[-1].name
    print(f"No v18 releases found — falling back to: {default}")
else:
    print("⚠ No submissions in /content/playground-s6e5/submissions/")
    print("  Run cell 6 (progressive blend) first.")
    print("  Or check Drive: MyDrive/Colab Notebooks/kaggle/s6e5/submissions/")
    default = None

# Override here if you want a specific release (e.g., 'v18.003.csv')
SUB_TO_DOWNLOAD = default

if SUB_TO_DOWNLOAD and Path(f'submissions/{SUB_TO_DOWNLOAD}').exists():
    print(f"\nDownloading: submissions/{SUB_TO_DOWNLOAD}")
    print("Submit from local with:")
    print(f"  kaggle competitions submit -c playground-series-s6e5 \\")
    print(f"      -f submissions/{SUB_TO_DOWNLOAD} -m 'v18 progressive blend'")
    files.download(f'submissions/{SUB_TO_DOWNLOAD}')
elif SUB_TO_DOWNLOAD is None:
    pass  # already printed message above
else:
    print(f"\nFile not found: submissions/{SUB_TO_DOWNLOAD}")
    print("Edit SUB_TO_DOWNLOAD above to a valid filename from the list.")